In [2]:
import openai
import instructor
from qdrant_client import QdrantClient

from pydantic import BaseModel, Field

In [6]:
prompt = """
You are helpful assistant.
Return answer to the question.
Question: What is your name?
"""

In [ ]:
openai_reponse = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "system", "content": prompt}],
    temperature=0,
)

print(openai_reponse.choices[0].message.content)

I am ChatGPT, your AI assistant. How can I help you today?


### Instructor Structured Outputs

In [3]:
client = instructor.from_openai(openai.OpenAI())

In [4]:
class RAGModelResponse(BaseModel):
    answer:str = Field(description="Answer to the question")

In [7]:
response, raw_reponse = client.chat.completions.create_with_completion(
    model="gpt-4.1-mini",
    messages=[{"role": "system", "content": prompt}],
    temperature=0,
    response_model=RAGModelResponse
)

In [11]:
response

RAGModelResponse(answer='My name is ChatGPT. How can I assist you today?')

In [9]:
raw_reponse

ChatCompletion(id='chatcmpl-DkSX5CPgBzPiBfmKiDMN5FEuMzzet', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_J65Qy5sfFl0x54n8CF52p54S', function=Function(arguments='{"answer":"My name is ChatGPT. How can I assist you today?"}', name='RAGModelResponse'), type='function')]))], created=1779965139, model='gpt-4.1-mini-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_59b5bbb72f', usage=CompletionUsage(completion_tokens=17, prompt_tokens=92, total_tokens=109, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

### RAG Example

In [12]:
class RAGModelResponse(BaseModel):
    answer:str = Field(description="Answer to the question")

In [13]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model,
    )

    return response.data[0].embedding


def retrieve_data(query, qdrant_client, top_k=5):
    query_embedding = get_embedding(query)
    search_result = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=top_k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    for point in search_result.points:
        retrieved_context_ids.append(point.payload["parent_asin"])
        retrieved_context.append(point.payload["description"])
        similarity_scores.append(point.score)
        retrieved_context_ratings.append(point.payload["average_rating"])
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"-ID: {id}, Rating: {rating}, Description: {chunk}\n"

    return formatted_context



def build_prompt(preprocessed_context, question):
    prompt = f"""
you are a shopping assistant that can answer questions about products in stock.

You will be given a question and a list of context. 

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as avaialable products.

Context:
{preprocessed_context}

Question: {question}
"""
    return prompt


def generate_answer(prompt):
    response, raw_response = client.chat.completions.create_with_completion(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": prompt},
        ],
        temperature=0,
        response_model=RAGModelResponse
        # reasoning_effort="minimal"
        # temperature=0.7,
        # max_tokens=500,
    )

    return response 



def rag_pipeline(question, qdrant_client, top_k=5):

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)
    final_result = {
        "data_model":answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context": retrieved_context["retrieved_context"],
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "similarity_scores": retrieved_context["similarity_scores"],
    }

    return final_result

In [14]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [17]:
output = rag_pipeline(question="What kind of Charging cables can i get? suggest me a good one", qdrant_client=qdrant_client)

In [18]:
output

{'data_model': RAGModelResponse(answer='You can get the Bauhoo Charging Station for Apple Multiple Devices, which is a 3 in 1 Fast Wireless Charger compatible with various iPhone models, Apple Watch, and AirPods. It is foldable and comes with an adapter, available in pink. This is a good option if you are looking for a wireless charging solution for multiple Apple devices.'),
 'answer': 'You can get the Bauhoo Charging Station for Apple Multiple Devices, which is a 3 in 1 Fast Wireless Charger compatible with various iPhone models, Apple Watch, and AirPods. It is foldable and comes with an adapter, available in pink. This is a good option if you are looking for a wireless charging solution for multiple Apple devices.',
 'question': 'What kind of Charging cables can i get? suggest me a good one',
 'retrieved_context': ['Bauhoo Charging Station for Apple Multiple Devices, 3 in 1 Fast Wireless Charger Foldable iPhone 14/13/12/11/Pro/XS/Xs Max/XR/X/SE/8/8 Plus Watch 8/7/6/SE/5/4/3/2 AirPod

In [19]:
print(output["answer"])

You can get the Bauhoo Charging Station for Apple Multiple Devices, which is a 3 in 1 Fast Wireless Charger compatible with various iPhone models, Apple Watch, and AirPods. It is foldable and comes with an adapter, available in pink. This is a good option if you are looking for a wireless charging solution for multiple Apple devices.


### RAG Pipeline with Grounded Context

In [37]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="ID of the item used to answer the question")
    description: str = Field(description="Short Description of the item used to answer the question")

class RAGModelResponseStructured(BaseModel):
    answer:str = Field(description="Answer to the question")
    references: list[RAGUsedContext] = Field(description="List of items used to answer the question")

In [38]:
def generate_answer_structured(prompt):
    response, raw_response = client.chat.completions.create_with_completion(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": prompt},
        ],
        temperature=0,
        response_model=RAGModelResponseStructured
        # reasoning_effort="minimal"
        # temperature=0.7,
        # max_tokens=500,
    )

    return response 

def build_prompt_structured(preprocessed_context, question):
    prompt = f"""
you are a shopping assistant that can answer questions about products in stock.

You will be given a question and a list of context. 

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as avaialable products.
- As an output you need to provide

* answer of the question based on provided context.
* list of the IDs of the chunks used to answer the question. Only return the ones that are used in the answer.
* short description (1-2 sentences) of the item based on the description provided in the context.

- the short description should have the name of the item.
- the answer to the question should contain detailed information about the product and returned with detailed specifications of the product in bullet points.

Context:
{preprocessed_context}

Question: {question}
"""
    return prompt


def rag_pipeline_stuctured(question, qdrant_client, top_k=5):

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt_structured(preprocessed_context, question)
    answer = generate_answer_structured(prompt)
    final_result = {
        "original_output":answer,
        "answer": answer.answer,
        "references":answer.references,
        "question": question,
        "retrieved_context": retrieved_context["retrieved_context"],
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "similarity_scores": retrieved_context["similarity_scores"],
    }

    return final_result

In [39]:
result = rag_pipeline_stuctured("Can i get some speakers?",qdrant_client,top_k=10)

In [40]:
result

{'original_output': RAGModelResponseStructured(answer='Yes, you can get several types of speakers from the available products:\n\n1. ABRRU Dual 5W Single USB Computer Speakers:\n- Two 3-inch all-magnetic speakers for clear sound\n- USB plug and play with no driver needed\n- Volume control and mute button\n- Compatible with Windows, macOS, Chrome OS, and USB-C devices\n- Includes USB-C to USB adapter\n- 1-year warranty and 2 months unconditional refund\n\n2. KENKUO Bluetooth Speakers with Lights:\n- 10W stereo audio driver with 360° surround sound\n- Bluetooth 5.2 with TWS mode for stereo pairing\n- Built-in FM radio, MicroSD card slot, and Aux-in\n- 1800mAh battery for 8-9 hours playtime\n- IPX6 waterproof rating\n- LED light show with 8 color themes\n\n3. Monster Boomerang Neck Speaker Bluetooth Wireless:\n- Qualcomm chip with apt-X for 3D stereo sound\n- Built-in 3W speaker system with bass diaphragms\n- 12 hours battery life\n- Ergonomic, lightweight neckband design\n- Hands-free ca

In [41]:
print(result["answer"])

Yes, you can get several types of speakers from the available products:

1. ABRRU Dual 5W Single USB Computer Speakers:
- Two 3-inch all-magnetic speakers for clear sound
- USB plug and play with no driver needed
- Volume control and mute button
- Compatible with Windows, macOS, Chrome OS, and USB-C devices
- Includes USB-C to USB adapter
- 1-year warranty and 2 months unconditional refund

2. KENKUO Bluetooth Speakers with Lights:
- 10W stereo audio driver with 360° surround sound
- Bluetooth 5.2 with TWS mode for stereo pairing
- Built-in FM radio, MicroSD card slot, and Aux-in
- 1800mAh battery for 8-9 hours playtime
- IPX6 waterproof rating
- LED light show with 8 color themes

3. Monster Boomerang Neck Speaker Bluetooth Wireless:
- Qualcomm chip with apt-X for 3D stereo sound
- Built-in 3W speaker system with bass diaphragms
- 12 hours battery life
- Ergonomic, lightweight neckband design
- Hands-free calls with voice pickup
- IPX7 waterproof

4. Bluetooth Bone Conduction Sport He